# 12. [LEAK] KOBIS 누적 관객수 사용

> ⚠️ **정답(최종 관객수)에 거의 가까운 값을 쓰는 실험**
> 대회 규칙("공공 데이터와 같이 누구나 얻을 수 있고 법적 제약이 없는 외부 데이터 허용")상 허용이지만, **모델링 실력과 무관한 점수**.
> 일반 결과(11-2, Public 339,580)와 분리해서 기록하고, 과제 최종 제출에 쓸지는 운영진 확인 후 결정.

**데이터**: `kobis_fetch_audience_leak.py` → KOBIS 일별 박스오피스 상위 10위의 누적 관객수(`audiAcc`) → 영화별 최댓값 `aud_acc_max`
(상위 10위 밖으로 밀려난 뒤의 관객은 빠지므로 최종 관객수보다 약간 작음)

| 방법 | 내용 |
| --- | --- |
| L1 | 11-2 모델(TabPFN)에 `aud_acc_max`를 변수로 추가 |
| L2 | 규칙: 매칭된 영화는 `aud_acc_max × 보정 배율`, 나머지는 11-2 예측 |

## 0. 설정 · 데이터

In [1]:
import os
import re
import glob
import json
import time
import pickle
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

from features import make_features

warnings.filterwarnings('ignore')
train = pd.read_csv('movies_train.csv')
test = pd.read_csv('movies_test.csv')
submission = pd.read_csv('submission.csv')
kobis = pd.read_csv('kobis_movies.csv', dtype={'movieCd': str})
aud = pd.read_csv('kobis_audience_leak.csv', dtype={'movieCd': str})
y = train['box_off_num'].values.astype(float)
hit = y >= 1_000_000
rmse = lambda a, b: mean_squared_error(a, b) ** 0.5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)
CACHE_DIR = 'cache_12_leak'
os.makedirs(CACHE_DIR, exist_ok=True)
print('누적 관객수가 있는 영화', len(aud), '편')

C:\Users\taehe\OneDrive\문서\대학교\3학년 2학기\ESAA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


누적 관객수가 있는 영화 1371 편


## 1. 11-2 변수 다시 만들기 (09 변수 + 흥행 기간 변수, 순위 기반 제외)

In [2]:
rows = []
for p in glob.glob('kobis_cache/*.json'):
    d = os.path.basename(p)[:8]
    for r in json.load(open(p, encoding='utf-8')):
        rows.append({**r, 'targetDt': pd.Timestamp(d)})
daily = pd.DataFrame(rows)
daily['movieCd'] = daily['movieCd'].astype(str)
for c in ['rank', 'scrnCnt', 'showCnt']:
    daily[c] = pd.to_numeric(daily[c], errors='coerce')
daily['open_dt'] = pd.to_datetime(daily['openDt'].str.strip().replace('', None), errors='coerce')
daily['day'] = (daily['targetDt'] - daily['open_dt']).dt.days


def traj(g):
    g = g[g['day'] >= 0]
    on = lambda n, col: g.loc[g['day'] == n, col].max() if (g['day'] == n).any() else 0.0
    open_scrn = g.loc[g['day'] == g['day'].min(), 'scrnCnt'].max() if len(g) else np.nan
    out = {'t_scrn_d7': on(7, 'scrnCnt'), 't_scrn_d14': on(14, 'scrnCnt'), 't_scrn_d21': on(21, 'scrnCnt'),
           't_show_d7': on(7, 'showCnt'), 't_show_d14': on(14, 'showCnt'),
           't_sum_scrn': g['scrnCnt'].sum(), 't_sum_show': g['showCnt'].sum()}
    out['t_keep_d7'] = out['t_scrn_d7'] / open_scrn if open_scrn else 0.0
    out['t_keep_d14'] = out['t_scrn_d14'] / open_scrn if open_scrn else 0.0
    return pd.Series(out)


T = daily.groupby('movieCd').apply(traj).fillna(0)
AUD = aud.set_index('movieCd')['aud_acc_max']
norm = lambda s: re.sub(r'[^0-9a-zA-Z가-힣]', '', str(s)).lower()
kobis['key'] = kobis['movieNm'].map(norm)
kobis['open_dt'] = pd.to_datetime(kobis['openDt'], errors='coerce')
K_COLS = ['open_scrn', 'open_show', 'max_scrn', 'max_show', 'days_top10', 'best_rank']


def kobis_features(df):
    out = []
    for title, r in zip(df['title'], pd.to_datetime(df['release_time'])):
        cand = kobis[kobis['key'] == norm(title)]
        row = {f'k_{c}': np.nan for c in K_COLS} | {c: np.nan for c in T.columns} | {'aud_acc_max': np.nan}
        if len(cand):
            gap = (cand['open_dt'] - r).abs().dt.days.fillna(9999)
            if gap.min() <= 30:
                best = cand.loc[gap.idxmin()]
                row = ({f'k_{c}': best[c] for c in K_COLS} | (T.loc[best['movieCd']].to_dict() if best['movieCd'] in T.index else {})
                       | {'aud_acc_max': AUD.get(best['movieCd'], np.nan)})
        out.append(row)
    K = pd.DataFrame(out)
    K['k_in_top10'] = K['k_max_scrn'].notna().astype(int)
    K['k_best_rank'] = K['k_best_rank'].fillna(11)
    return K


base_tr, base_te = make_features(train, test, {'debut': 1, 'debut_mode': 'nan', 'distributor': 1, 'date': 1, 'rating': 1})
KT_tr, KT_te = kobis_features(train), kobis_features(test)
aud_tr, aud_te = KT_tr.pop('aud_acc_max').values, KT_te.pop('aud_acc_max').values
KT_tr, KT_te = KT_tr.fillna(0).astype(float), KT_te.fillna(0).astype(float)
X112 = pd.concat([base_tr.reset_index(drop=True), KT_tr], axis=1)
X112_te = pd.concat([base_te.reset_index(drop=True), KT_te], axis=1)
print('11-2 변수', X112.shape[1], '개 | 누적 관객수 매칭 train', int(np.isfinite(aud_tr).sum()), '/ test', int(np.isfinite(aud_te).sum()))

11-2 변수 57 개 | 누적 관객수 매칭 train 275 / test 111


## 2. 누적 관객수 vs 실제 관객수

In [3]:
m = np.isfinite(aud_tr) & (aud_tr > 0)
ratio = y[m] / aud_tr[m]
print(f'실제 / 누적(상위 10위 기간) 배율: 중앙값 {np.median(ratio):.3f} · 평균 {ratio.mean():.3f} · 10~90% {np.percentile(ratio,10):.3f}~{np.percentile(ratio,90):.3f}')
print(f'매칭 영화의 Spearman: {pd.Series(aud_tr[m]).corr(pd.Series(y[m]), method="spearman"):.4f}')
big = m & hit
print(f'흥행작(100만+)만: 배율 중앙값 {np.median(y[big]/aud_tr[big]):.3f}')

실제 / 누적(상위 10위 기간) 배율: 중앙값 1.011 · 평균 1.297 · 10~90% 1.001~1.441
매칭 영화의 Spearman: 0.9988
흥행작(100만+)만: 배율 중앙값 1.003


## 3. L2: 규칙 (매칭 영화 = 누적 × 배율, 나머지 = 11-2 예측)

In [4]:
p112 = pickle.load(open('cache_11_2/T2_흐름_순위제외.pkl', 'rb'))
p112_oof, p112_test = p112['oof'], p112['test']

# 배율은 fold 안에서 train 부분으로만 계산 (누수 방지)
l2_oof = p112_oof.copy()
for tr_idx, va_idx in kf5.split(train):
    mt = np.isfinite(aud_tr[tr_idx]) & (aud_tr[tr_idx] > 0)
    k = np.median(y[tr_idx][mt] / aud_tr[tr_idx][mt])
    mv = np.isfinite(aud_tr[va_idx]) & (aud_tr[va_idx] > 0)
    l2_oof[va_idx[mv]] = aud_tr[va_idx][mv] * k
k_all = np.median(ratio)
l2_test = p112_test.copy()
mt = np.isfinite(aud_te) & (aud_te > 0)
l2_test[mt] = aud_te[mt] * k_all
print(f'L2 CV RMSE {rmse(y, l2_oof):,.0f} | 흥행작 {rmse(y[hit], l2_oof[hit]):,.0f}  (11-2: {rmse(y, p112_oof):,.0f})')

L2 CV RMSE 43,114 | 흥행작 39,974  (11-2: 290,485)


## 4. L1: 11-2 TabPFN + 누적 관객수 변수

In [5]:
path = os.path.join(CACHE_DIR, 'L1.pkl')
if os.path.exists(path):
    L1 = pickle.load(open(path, 'rb'))
else:
    X = X112.assign(aud_acc_max=np.nan_to_num(aud_tr), aud_in=np.isfinite(aud_tr).astype(float))
    Xt = X112_te.assign(aud_acc_max=np.nan_to_num(aud_te), aud_in=np.isfinite(aud_te).astype(float))
    mk = lambda: TabPFNRegressor.create_default_for_version(ModelVersion.V3_5, device=DEVICE, random_state=42)
    oof = np.zeros(len(y))
    for tr_idx, va_idx in kf5.split(X):
        mdl = mk(); mdl.fit(X.iloc[tr_idx], np.sqrt(y[tr_idx]))
        oof[va_idx] = np.square(np.clip(mdl.predict(X.iloc[va_idx]), 0, None))
    mdl = mk(); mdl.fit(X, np.sqrt(y))
    L1 = {'oof': oof, 'test': np.square(np.clip(mdl.predict(Xt), 0, None))}
    pickle.dump(L1, open(path, 'wb'))
print(f"L1 CV RMSE {rmse(y, L1['oof']):,.0f} | 흥행작 {rmse(y[hit], L1['oof'][hit]):,.0f}")

L1 CV RMSE 47,870 | 흥행작 63,326


## 5. 비교 · 제출 파일 (파일 이름에 `leak` 표시)

In [6]:
res = pd.DataFrame([
    {'방법': '11-2 (누적 관객수 없음) · Public 339,580', 'CV RMSE': rmse(y, p112_oof), '흥행작 RMSE': rmse(y[hit], p112_oof[hit])},
    {'방법': 'L1: TabPFN + 누적 관객수', 'CV RMSE': rmse(y, L1['oof']), '흥행작 RMSE': rmse(y[hit], L1['oof'][hit])},
    {'방법': 'L2: 규칙 (누적 × 배율)', 'CV RMSE': rmse(y, l2_oof), '흥행작 RMSE': rmse(y[hit], l2_oof[hit])},
])
display(res.style.format({'CV RMSE': '{:,.0f}', '흥행작 RMSE': '{:,.0f}'}).highlight_min(subset=['CV RMSE'], color='#c6efce'))

for name, pred in [('12_leak_L1_tabpfn.csv', L1['test']), ('12_leak_L2_rule.csv', l2_test)]:
    sub = submission.copy(); sub['box_off_num'] = np.clip(pred, 0, None)
    sub.to_csv(name, index=False)
    print('저장:', name)

,방법,CV RMSE,흥행작 RMSE
0,"11-2 (누적 관객수 없음) · Public 339,580","290,485","664,424"
1,L1: TabPFN + 누적 관객수,"47,870","63,326"
2,L2: 규칙 (누적 × 배율),"43,114","39,974"


저장: 12_leak_L1_tabpfn.csv
저장: 12_leak_L2_rule.csv
